In [1]:
# Copyright (c) 2024, NVIDIA CORPORATION.
"""Pretrain Vision Transformer using Megatron Core."""
import os
import math
import torch
import torch.nn.functional as F
from torch import Tensor

from megatron.core import parallel_state, tensor_parallel
from megatron.core.enums import ModelType
from megatron.core.transformer.transformer_block import TransformerBlock
from megatron.core.transformer.transformer_config import TransformerConfig
from megatron.core.transformer.module import GraphableMegatronModule, MegatronModule
from megatron.core.models.vision.vit_layer_specs import (
    get_vit_layer_with_local_spec,
    get_vit_layer_with_transformer_engine_spec,
)
from megatron.core.models.common.embeddings import RotaryEmbeddingDinoV3

from megatron.training import (
    get_args,
    get_timers,
    pretrain,
    print_rank_0,
)
from megatron.training.arguments import core_transformer_config_from_args

from megatron.core.datasets.blended_megatron_dataset_builder import (
    BlendedMegatronDatasetBuilder,
)
from megatron.core.datasets.utils import Split
from megatron.core.datasets.blended_megatron_dataset_config import (
    BlendedMegatronDatasetConfig,
)
from megatron.training import initialize_megatron


import sys, shlex

with open("debug_args.txt") as f:
    args_list = shlex.split(f.read())

sys.argv = ["notebook"] + args_list


/usr/local/lib/python3.12/dist-packages/torch/library.py:356: UserWarning: Warning only once for all operators,  other operators may also be overridden.
  Overriding a previously registered kernel for the same operator and the same dispatch key
  operator: flash_attn::_flash_attn_backward(Tensor dout, Tensor q, Tensor k, Tensor v, Tensor out, Tensor softmax_lse, Tensor(a6!)? dq, Tensor(a7!)? dk, Tensor(a8!)? dv, float dropout_p, float softmax_scale, bool causal, SymInt window_size_left, SymInt window_size_right, float softcap, Tensor? alibi_slopes, bool deterministic, Tensor? rng_state=None) -> Tensor
    registered at /usr/local/lib/python3.12/dist-packages/torch/_library/custom_ops.py:922
  dispatch key: ADInplaceOrView
  previous kernel: no debug info
       new kernel: registered at /usr/local/lib/python3.12/dist-packages/torch/_library/custom_ops.py:922 (Triggered internally at /opt/pytorch/pytorch/aten/src/ATen/core/dispatch/OperatorEntry.cpp:208.)
  self.m.impl(
/usr/local/lib/p

In [2]:
from pretrain_vision import *

In [3]:
initialize_megatron(
        extra_args_provider=add_vit_args,
        args_defaults={},
        get_embedding_ranks=None,
        get_position_embedding_ranks=None,
        store=None,
    )

using world size: 1, data-parallel size: 1, context-parallel size: 1, hierarchical context-parallel sizes: None, tensor-model-parallel size: 1, pipeline-model-parallel size: 1
Number of virtual stages per pipeline stage: None
accumulate and all-reduce gradients in fp32 for bfloat16 data type.
using torch.bfloat16 for parameters ...
------------------------ arguments ------------------------
  account_for_embedding_in_pipeline_split ......... False
  account_for_loss_in_pipeline_split .............. False
  accumulate_allreduce_grads_in_fp32 .............. True
  activation_func_clamp_value ..................... None
  adam_beta1 ...................................... 0.9
  adam_beta2 ...................................... 0.95
  adam_eps ........................................ 1e-08
  add_bias_linear ................................. True
  add_position_embedding .......................... True
  add_qkv_bias .................................... True
  adlr_autoresume ................

  ddp_num_buckets ................................. None
  ddp_pad_buckets_for_high_nccl_busbw ............. False
  ddp_reduce_scatter_with_fp32_accumulation ....... False
  decode_only_cuda_graphs ......................... False
  decoder_first_pipeline_num_layers ............... None
  decoder_last_pipeline_num_layers ................ None
  decoder_num_layers .............................. None
  decoder_seq_length .............................. None
  decoupled_lr .................................... None
  decoupled_min_lr ................................ None
  decrease_batch_size_if_needed ................... False
  defer_embedding_wgrad_compute ................... False
  delay_wgrad_compute ............................. False
  deprecated_use_mcore_models ..................... False
  deterministic_mode .............................. False
  dino_bottleneck_size ............................ 256
  dino_freeze_last_layer .......................... 1
  dino_head_hidden_size ...

/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W220 17:47:22.126761561 ProcessGroupNCCL.cpp:5156] Guessing device ID based on global rank. This can cause a hang if rank to GPU mapping is heterogeneous. You can specify device_id in init_process_group()


In [4]:
model = model_provider(pre_process=True, post_process=True,config=None,pg_collection=None,wrap_with_ddp=False)

Building ViT model ...


height:512, width:512




patch size: 16



dim at rope: 64


periods size: torch.Size([16])



/workspace/megatron-lm/megatron/core/transformer/transformer_config.py:1189: UserWarning: If you are using transformer_engine as the transformer implementation, the core_attn is from transformer_engine and may be the fused version. For fused attention, you have no need to set 'core_attn' to recompute. Please check that the core_attn recompute is really needed.
  warnings.warn(


In [5]:
train_ds, valid_ds, test_ds = train_valid_test_datasets_provider()

Building ViT datasets with BlendedMegatronDatasetBuilder ...
INFO:megatron.core.datasets.blended_megatron_dataset_config:blend not provided for test split
INFO:megatron.core.datasets.blended_megatron_dataset_builder:Building MegatronVisionDataset splits with sizes=[20, 1, 1] and config=BlendedMegatronDatasetConfig(random_seed=1234, sequence_length=0, image_size=512, blend=None, blend_per_split=[(['/workspace/dataset/training'], None), (['/workspace/dataset/testing'], None), None], multiple_validation_sets=None, full_validation=None, split=None, split_matrix=None, num_dataset_builder_threads=1, path_to_cache=None, mmap_bin_files=True, mock=False, tokenizer=None, mid_level_dataset_surplus=0.005, allow_ambiguous_pad_tokens=False, fast_cache_load=False, defer_npy_index_mmap=False)


In [6]:
image = train_ds[0]["images"]
label = train_ds[0]["labels"]
image = torch.unsqueeze(image, 0)
image = image.to("cuda",dtype=torch.bfloat16)
label = label.to("cuda")



In [7]:
image.to("cuda",)

tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]]], device='cuda:0',
       dtype=torch.bfloat16)

In [8]:
from pretrain_vision import *
model = model_provider(pre_process=True, post_process=True,config=None,pg_collection=None,wrap_with_ddp=False)
meg_model = model.to("cuda",dtype=torch.bfloat16)
meg_model(image)

Building ViT model ...


height:512, width:512




patch size: 16



dim at rope: 64


periods size: torch.Size([16])



x size: torch.Size([1, 384, 32, 32])



angles size: torch.Size([1024, 2, 16])


angles size after flatten: torch.Size([1024, 32])


angles sizeafter branch: torch.Size([1024, 64])



 embed size: torch.Size([1024, 128])




pos_emb: torch.Size([1024, 1, 1, 64]), x: torch.Size([1025, 1, 384])


q_pos_emb size: torch.Size([1024, 1, 1, 64]), k_pos_emb size: torch.Size([1024, 1, 1, 64])


query shape: torch.Size([1025, 1, 6, 64]), num_prefix_tokens: 1, num_pos_tokens: 1024


query shape after split: torch.Size([1024, 1, 6, 64]), pos_emb size: torch.Size([1024, 1, 1, 64])
q_pos_emb size: torch.Size([1024, 1, 1, 64]), k_pos_emb size: torch.Size([1024, 1, 1, 64])


query shape: torch.Size([1025, 1, 6, 64]), num_prefix_tokens: 1, num_pos_tokens: 1024


query shape after split: torch.Size([1024, 1, 6, 64]), pos_emb size: torch.Size([1024, 1, 1, 64])
q_pos_emb size: torch.Siz

tensor([[ 0.2373,  0.2412,  1.5859, -0.1934]], device='cuda:0',
       dtype=torch.bfloat16, grad_fn=<AddmmBackward0>)

In [9]:
meg_model(image)



x size: torch.Size([1, 384, 32, 32])



angles size: torch.Size([1024, 2, 16])


angles size after flatten: torch.Size([1024, 32])


angles sizeafter branch: torch.Size([1024, 64])



 embed size: torch.Size([1024, 128])




pos_emb: torch.Size([1024, 1, 1, 64]), x: torch.Size([1025, 1, 384])


q_pos_emb size: torch.Size([1024, 1, 1, 64]), k_pos_emb size: torch.Size([1024, 1, 1, 64])


query shape: torch.Size([1025, 1, 6, 64]), num_prefix_tokens: 1, num_pos_tokens: 1024


query shape after split: torch.Size([1024, 1, 6, 64]), pos_emb size: torch.Size([1024, 1, 1, 64])
q_pos_emb size: torch.Size([1024, 1, 1, 64]), k_pos_emb size: torch.Size([1024, 1, 1, 64])


query shape: torch.Size([1025, 1, 6, 64]), num_prefix_tokens: 1, num_pos_tokens: 1024


query shape after split: torch.Size([1024, 1, 6, 64]), pos_emb size: torch.Size([1024, 1, 1, 64])
q_pos_emb size: torch.Size([1024, 1, 1, 64]), k_pos_emb size: torch.Size([1024, 1, 1, 64])


query shape: torch.Size([1025, 1, 6, 64]), num_pref

tensor([[0.7422, 0.0486, 0.6133, 0.5078]], device='cuda:0',
       dtype=torch.bfloat16, grad_fn=<AddmmBackward0>)

In [10]:
import torch
from transformers import AutoImageProcessor, AutoModel
from transformers.image_utils import load_image

pretrained_model_name = "facebook/dinov3-vits16-pretrain-lvd1689m"
processor = AutoImageProcessor.from_pretrained(pretrained_model_name)
model = AutoModel.from_pretrained(
    pretrained_model_name, 
    device_map="auto", 
)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m/resolve/main/preprocessor_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m/resolve/main/preprocessor_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m/resolve/main/config.json "HTTP/1.1 200 OK"

dim at rope: 64


inv_freq size: torch.Size([16])



Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

In [11]:
out = model(image)
print(model)

64

angles size: torch.Size([1024, 2, 16])


angles size after flatten: torch.Size([1024, 32])


angles size after tile: torch.Size([1024, 64])



 cos size: torch.Size([1024, 64]), sin size: torch.Size([1024, 64])




pos emb sizes: torch.Size([1024, 64]), torch.Size([1024, 64])
pos emb sizes: torch.Size([1024, 64]), torch.Size([1024, 64])
pos emb sizes: torch.Size([1024, 64]), torch.Size([1024, 64])
pos emb sizes: torch.Size([1024, 64]), torch.Size([1024, 64])
pos emb sizes: torch.Size([1024, 64]), torch.Size([1024, 64])
pos emb sizes: torch.Size([1024, 64]), torch.Size([1024, 64])
pos emb sizes: torch.Size([1024, 64]), torch.Size([1024, 64])
pos emb sizes: torch.Size([1024, 64]), torch.Size([1024, 64])
pos emb sizes: torch.Size([1024, 64]), torch.Size([1024, 64])
pos emb sizes: torch.Size([1024, 64]), torch.Size([1024, 64])
pos emb sizes: torch.Size([1024, 64]), torch.Size([1024, 64])
pos emb sizes: torch.Size([1024, 64]), torch.Size([1024, 64])
DINOv3ViTModel(
  (embeddings): DINOv3ViTEmbeddings(
    (patch_embeddings): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
  )
  (rope_embeddings): DINOv3ViTRopePositionEmbedding()
  (layer): ModuleList(
    (0-11): 12 x DINOv3ViTLayer(
      (norm

In [ ]:
pos_emb_hf = torch.load("/workspace/megatron-lm/hf_emb_tensor")
pos_emb_mlm = torch.load("/workspace/megatron-lm/mlm_emb_tensor")